# FTIR ML Solver v3 — Colab Training

## Before running
1. **Runtime → Change runtime type → GPU (T4)**
2. Run cells top to bottom

In [ ]:
# Cell 1 — Download repo and install
import urllib.request, zipfile, os, shutil, sys

url = 'https://github.com/DrSuppe/FTIR_absorbtion_ML/archive/refs/heads/main.zip'
print('Downloading...')
urllib.request.urlretrieve(url, '/content/repo.zip')

print('Extracting...')
with zipfile.ZipFile('/content/repo.zip') as z:
    z.extractall('/content/')

extracted = '/content/FTIR_absorbtion_ML-main'
target    = '/content/ftir'
if os.path.exists(target):
    shutil.rmtree(target)
os.rename(extracted, target)
os.chdir(target)
print('Working directory:', os.getcwd())

# Install WITHOUT overwriting Colab's pre-installed torch/numpy
# (do NOT use requirements-pinned.txt — it has versions that conflict with Colab)
!pip install . --no-deps -q

# Colab kernels don't always refresh sys.path immediately after pip install.
# Adding src/ to sys.path guarantees it imports immediately.
sys.path.insert(0, '/content/ftir/src')

# Verify
from ftir_analysis.constants import DEFAULT_TARGET_SPECIES
print('ftir_analysis installed OK')
print('Target species:', DEFAULT_TARGET_SPECIES)

In [ ]:
# Cell 2 — Build / Rebuild Manifest
from ftir_analysis.manifesting import build_manifest
m = build_manifest()
print(f"Manifest: {len(m)} spectra, {m.species.nunique()} unique species")

In [ ]:
# Cell 3 — Run A control (~1-2h, hybrid_v4 trace_frac=0.15)
!python3 synthetic_generator.py \
    --n-samples 20000 \
    --seed 42 \
    --sampling-mode hybrid_v4 \
    --augmentation-profile mild \
    --hybrid-trace-fraction 0.15 \
    --min-active-species 1 \
    --max-active-species 4 \
    --out data/synthetic/spectra_v4_base.npz \
    --diagnostics-json outputs/reports/run_a/generation_diagnostics.json

!python3 train.py \
    --n-synthetic 20000 \
    --synthetic-npz data/synthetic/spectra_v4_base.npz \
    --epochs 16 \
    --batch-size 64 \
    --reference-weight 0.20 \
    --active-label-weight 4.0 \
    --inactive-label-weight 0.5 \
    --warmup-epochs 2.0 \
    --synthetic-sampling-mode hybrid_v4 \
    --synthetic-augmentation-profile mild \
    --hybrid-trace-fraction 0.15 \
    --min-active-species 1 \
    --max-active-species 4 \
    --checkpoint-dir outputs/checkpoints/run_a \
    --reports-dir outputs/reports/run_a \
    --log-every 2

In [ ]:
# Cell 4 — Run B prior model (~1-2h, same data + light spectroscopy priors)

!python3 train.py \
    --n-synthetic 20000 \
    --synthetic-npz data/synthetic/spectra_v4_base.npz \
    --epochs 16 \
    --batch-size 64 \
    --reference-weight 0.20 \
    --active-label-weight 4.0 \
    --inactive-label-weight 0.5 \
    --warmup-epochs 2.0 \
    --synthetic-sampling-mode hybrid_v4 \
    --synthetic-augmentation-profile mild \
    --hybrid-trace-fraction 0.15 \
    --min-active-species 1 \
    --max-active-species 4 \
    --use-prior-features \
    --checkpoint-dir outputs/checkpoints/run_b \
    --reports-dir outputs/reports/run_b \
    --log-every 2

In [ ]:
# Cell 5 — Run C prior model + lighter trace emphasis (~1-2h, hybrid_v4 trace_frac=0.10)
!python3 synthetic_generator.py \
    --n-samples 20000 \
    --seed 42 \
    --sampling-mode hybrid_v4 \
    --augmentation-profile mild \
    --hybrid-trace-fraction 0.10 \
    --min-active-species 1 \
    --max-active-species 4 \
    --out data/synthetic/spectra_v4_trace10.npz \
    --diagnostics-json outputs/reports/run_c/generation_diagnostics.json

!python3 train.py \
    --n-synthetic 20000 \
    --synthetic-npz data/synthetic/spectra_v4_trace10.npz \
    --epochs 16 \
    --batch-size 64 \
    --reference-weight 0.20 \
    --active-label-weight 4.0 \
    --inactive-label-weight 0.5 \
    --warmup-epochs 2.0 \
    --synthetic-sampling-mode hybrid_v4 \
    --synthetic-augmentation-profile mild \
    --hybrid-trace-fraction 0.10 \
    --min-active-species 1 \
    --max-active-species 4 \
    --use-prior-features \
    --checkpoint-dir outputs/checkpoints/run_c \
    --reports-dir outputs/reports/run_c \
    --log-every 2

In [ ]:
# Cell 6 — Compare runs by held-out reference metric with major-species guardrail
import torch
import json
from pathlib import Path

def _find_file(rel_path: str) -> Path | None:
    rel = Path(rel_path)
    roots = [Path.cwd(), Path('/content/ftir'), Path('/content')]
    seen: set[Path] = set()
    for root in roots:
        if not root.exists():
            continue
        root = root.resolve()
        if root in seen:
            continue
        seen.add(root)
        candidate = root / rel
        if candidate.exists():
            return candidate
    if Path('/content').exists():
        hits = sorted(Path('/content').glob(f'**/{rel_path}'))
        if hits:
            return hits[0]
    return None

def _load_run(label: str, run_name: str) -> dict | None:
    ckpt = _find_file(f'outputs/checkpoints/{run_name}/best_model.pt')
    summary_path = _find_file(f'outputs/reports/{run_name}/training_summary.json')
    if ckpt is None:
        return None
    payload = torch.load(ckpt, map_location='cpu')
    summary = None
    if summary_path is not None:
        summary = json.loads(summary_path.read_text(encoding='utf-8'))
    selected = (summary.get('best_ref_epoch') or payload) if summary else payload
    best_mixed = summary.get('best_mixed_epoch') if summary else None
    return {
        'label': label,
        'run_name': run_name,
        'ckpt': ckpt,
        'selected': selected,
        'best_mixed': best_mixed,
    }

def _score(epoch_payload: dict) -> tuple[float, float, int]:
    raw_ref = epoch_payload.get('ref_val_log_mae_mean', float('inf'))
    raw_val = epoch_payload.get('val_log_mae_mean', float('inf'))
    ref = float(raw_ref if raw_ref is not None else float('inf'))
    val = float(raw_val if raw_val is not None else float('inf'))
    beat = int(epoch_payload.get('species_beating_zero_baseline', -1))
    return ref, val, beat

def _group(epoch_payload: dict, split: str, group: str) -> float:
    grouped = epoch_payload.get('group_log_mae_mean', {})
    split_payload = grouped.get(split) or {}
    return float(split_payload.get(group, float('inf')))

runs = [
    _load_run('A', 'run_a'),
    _load_run('B', 'run_b'),
    _load_run('C', 'run_c'),
]
runs = [r for r in runs if r is not None]

if len(runs) < 2:
    print('Run at least two sweep cells first. Available checkpoints:')
    for run_name in ('run_a', 'run_b', 'run_c'):
        print(run_name, _find_file(f'outputs/checkpoints/{run_name}/best_model.pt'))
else:
    control = next((r for r in runs if r['label'] == 'A'), None)
    control_major = _group(control['selected'], 'all', 'major') if control else float('inf')
    MARGINAL_REF_GAIN = 0.02
    MATERIAL_MAJOR_REGRESSION = 0.05
    print(f'Guardrail thresholds: ref gain >= {MARGINAL_REF_GAIN:.2f} or major-group regression <= {MATERIAL_MAJOR_REGRESSION:.2f}')
    print()
    scored = []
    for run in runs:
        sel = run['selected']
        ref, val, beat = _score(sel)
        major_all = _group(sel, 'all', 'major')
        trace_all = _group(sel, 'all', 'trace')
        major_ref = _group(sel, 'ref', 'major')
        trace_ref = _group(sel, 'ref', 'trace')
        print(f"Run {run['label']}: best_ref_epoch={sel.get('epoch', 'n/a')} ref={ref:.4f} mixed={val:.4f} beat_zero={beat}/11")
        print(f'  groups(all): major={major_all:.4f} trace={trace_all:.4f}')
        print(f'  groups(ref): major={major_ref:.4f} trace={trace_ref:.4f}')
        if run['best_mixed'] is not None:
            bm = run['best_mixed']
            bm_val = bm.get('val_log_mae_mean', float('inf'))
            bm_ref = bm.get('ref_val_log_mae_mean', float('inf'))
            bm_val = float(bm_val if bm_val is not None else float('inf'))
            bm_ref = float(bm_ref if bm_ref is not None else float('inf'))
            print(f"  best_mixed_epoch={bm.get('epoch', 'n/a')} mixed={bm_val:.4f} ref={bm_ref:.4f}")
        guardrail_ok = True
        if control is not None and run['label'] != 'A':
            ref_gain = float(control['selected'].get('ref_val_log_mae_mean', float('inf'))) - ref
            major_regression = major_all - control_major
            guardrail_ok = not (ref_gain < MARGINAL_REF_GAIN and major_regression > MATERIAL_MAJOR_REGRESSION)
            print(f'  vs control: ref_gain={ref_gain:+.4f} major_regression={major_regression:+.4f} guardrail_ok={guardrail_ok}')
        print()
        scored.append((guardrail_ok, ref, val, -beat, run))

    viable = [item for item in scored if item[0]] or scored
    _, best_ref, best_val, _, winner_run = min(viable, key=lambda item: (item[1], item[2], item[3]))
    print(f"Winner: Run {winner_run['label']} (ref={best_ref:.4f}, mixed={best_val:.4f}, primary=held-out reference log-MAE)")

In [ ]:
# Cell 7 — View the latest per-species MAE plot
from pathlib import Path
import matplotlib.pyplot as plt, matplotlib.image as mpimg

candidate_roots = [Path.cwd(), Path('/content/ftir'), Path('/content')]
plot_patterns = [
    'outputs/reports/run_a/mae_per_species_*.png',
    'outputs/reports/run_b/mae_per_species_*.png',
    'outputs/reports/run_c/mae_per_species_*.png',
    'outputs/reports/mae_per_species_*.png',
]

plots: list[Path] = []
seen: set[Path] = set()
for root in candidate_roots:
    if not root.exists():
        continue
    for pattern in plot_patterns:
        for p in root.glob(pattern):
            rp = p.resolve()
            if rp not in seen:
                seen.add(rp)
                plots.append(rp)

if not plots and Path('/content').exists():
    for p in Path('/content').glob('**/mae_per_species_*.png'):
        rp = p.resolve()
        if rp not in seen:
            seen.add(rp)
            plots.append(rp)

plots = sorted(plots, key=lambda p: p.stat().st_mtime)
if plots:
    latest = plots[-1]
    img = mpimg.imread(latest)
    plt.figure(figsize=(12, 5))
    plt.imshow(img)
    plt.axis('off')
    plt.title(f'MAE — {latest.parent.name}/{latest.name}')
    plt.show()
    print(f'Loaded: {latest}')
else:
    print('No MAE plots found. Run training first.')
    print(f'cwd={Path.cwd()}')

In [ ]:
# Cell 8 — Download best checkpoint
from pathlib import Path
import torch
import json
from google.colab import files

def _find_file(rel_path: str) -> Path | None:
    rel = Path(rel_path)
    roots = [Path.cwd(), Path('/content/ftir'), Path('/content')]
    seen: set[Path] = set()
    for root in roots:
        if not root.exists():
            continue
        root = root.resolve()
        if root in seen:
            continue
        seen.add(root)
        candidate = root / rel
        if candidate.exists():
            return candidate
    if Path('/content').exists():
        hits = sorted(Path('/content').glob(f'**/{rel_path}'))
        if hits:
            return hits[0]
    return None

def _load_selected(label: str, run_name: str) -> dict | None:
    ckpt = _find_file(f'outputs/checkpoints/{run_name}/best_model.pt')
    summary_path = _find_file(f'outputs/reports/{run_name}/training_summary.json')
    if ckpt is None:
        return None
    payload = torch.load(ckpt, map_location='cpu')
    summary = None
    if summary_path is not None:
        summary = json.loads(summary_path.read_text(encoding='utf-8'))
    selected = (summary.get('best_ref_epoch') or payload) if summary else payload
    return {'label': label, 'ckpt': ckpt, 'selected': selected}

def _score(epoch_payload: dict) -> tuple[float, float, int]:
    raw_ref = epoch_payload.get('ref_val_log_mae_mean', float('inf'))
    raw_val = epoch_payload.get('val_log_mae_mean', float('inf'))
    ref = float(raw_ref if raw_ref is not None else float('inf'))
    val = float(raw_val if raw_val is not None else float('inf'))
    beat = int(epoch_payload.get('species_beating_zero_baseline', -1))
    return ref, val, beat

def _group(epoch_payload: dict, split: str, group: str) -> float:
    grouped = epoch_payload.get('group_log_mae_mean', {})
    split_payload = grouped.get(split) or {}
    return float(split_payload.get(group, float('inf')))

runs = [
    _load_selected('A', 'run_a'),
    _load_selected('B', 'run_b'),
    _load_selected('C', 'run_c'),
]
runs = [r for r in runs if r is not None]
if not runs:
    raise FileNotFoundError(f'No run_a/run_b/run_c checkpoint found. cwd={Path.cwd()}')

control = next((r for r in runs if r['label'] == 'A'), None)
control_ref = float(control['selected'].get('ref_val_log_mae_mean', float('inf'))) if control else float('inf')
control_major = _group(control['selected'], 'all', 'major') if control else float('inf')
MARGINAL_REF_GAIN = 0.02
MATERIAL_MAJOR_REGRESSION = 0.05
scored = []
for run in runs:
    ref, val, beat = _score(run['selected'])
    guardrail_ok = True
    if control is not None and run['label'] != 'A':
        ref_gain = control_ref - ref
        major_regression = _group(run['selected'], 'all', 'major') - control_major
        guardrail_ok = not (ref_gain < MARGINAL_REF_GAIN and major_regression > MATERIAL_MAJOR_REGRESSION)
    scored.append((guardrail_ok, ref, val, -beat, run))

viable = [item for item in scored if item[0]] or scored
_, best_ref, best_val, _, winner = min(viable, key=lambda item: (item[1], item[2], item[3]))
print(f"Downloading winner Run {winner['label']}: {winner['ckpt']} (ref={best_ref:.4f}, mixed={best_val:.4f})")
files.download(str(winner['ckpt']))